# Final Verification — independent audit of the submission

This is the **last** notebook in the pipeline. It re-reads everything from disk and re-checks the chain of correctness:

1. `submission.csv` schema, row count, Index alignment, value range
2. No NaN / inf / out-of-range demand values
3. Distribution sanity vs train target by RoadType_modal and by qhour
4. Independent recomputation of the OOF blend weights (sanity vs blend_weights.json)
5. Leakage spot-check on test_features: modal features and Temperature imputations
6. Saturation correction fired on the correct number of rows

Each check prints **PASS / FAIL** plus the concrete number.


## 1. Setup

In [1]:
import os, json
import numpy as np
import pandas as pd

BASE = '.'
DATASET   = os.path.join(BASE, 'e88186124ec611f1 (1)', 'dataset')
PROCESSED = os.path.join(BASE, 'processed')
OOF_DIR   = os.path.join(BASE, 'oof')
PRED_DIR  = os.path.join(BASE, 'predictions')

raw_train = pd.read_csv(os.path.join(DATASET, 'train.csv'))
raw_test  = pd.read_csv(os.path.join(DATASET, 'test.csv'))
sub       = pd.read_csv(os.path.join(BASE, 'submission.csv'))

print('raw_train:', raw_train.shape, '| raw_test:', raw_test.shape, '| submission:', sub.shape)

raw_train: (77299, 11) | raw_test: (41778, 10) | submission: (41778, 2)


## 2. Submission schema + integrity

In [2]:
def check(name, cond, detail=''):
    print(f'{"PASS" if cond else "FAIL"}  {name}  {detail}')

check('sub row count == 41778',     len(sub) == 41778, f'(got {len(sub)})')
check('sub columns == [Index, demand]', list(sub.columns) == ['Index', 'demand'], f'(got {list(sub.columns)})')
check('sub Index dtype is integer', np.issubdtype(sub['Index'].dtype, np.integer), f'(got {sub["Index"].dtype})')
check('sub demand dtype is float',  np.issubdtype(sub['demand'].dtype, np.floating), f'(got {sub["demand"].dtype})')
check('sub Index aligns with raw_test row-by-row',
      (sub['Index'].values == raw_test['Index'].values).all())
check('sub has 0 NaN',  sub.isna().sum().sum() == 0)
check('sub has 0 inf',  np.isfinite(sub['demand']).all())
check('sub.demand in [0, 1]', (sub['demand'].min() >= 0) and (sub['demand'].max() <= 1),
      f'(range [{sub["demand"].min():.5f}, {sub["demand"].max():.5f}])')

PASS  sub row count == 41778  (got 41778)
PASS  sub columns == [Index, demand]  (got ['Index', 'demand'])
PASS  sub Index dtype is integer  (got int64)
PASS  sub demand dtype is float  (got float64)
PASS  sub Index aligns with raw_test row-by-row  
PASS  sub has 0 NaN  
PASS  sub has 0 inf  
PASS  sub.demand in [0, 1]  (range [0.00354, 0.99965])


## 3. Distribution sanity

In [3]:
# Mean demand by RoadType_modal in test prediction
test_feat = pd.read_parquet(os.path.join(PROCESSED, 'test_features.parquet'))
joined = sub.merge(test_feat[['Index','RoadType_modal','NumberofLanes_modal','qhour_of_day']], on='Index')

# Reference: train target by RoadType (from raw_train)
train_means = raw_train.groupby('RoadType')['demand'].mean()
pred_means  = joined.groupby('RoadType_modal')['demand'].mean()

print('Reference vs predicted mean by RoadType:')
print(pd.concat([train_means.rename('train_mean'),
                 pred_means.rename('pred_mean_test')],
                axis=1))
# Highway should be in the ballpark of the train mean (~0.6) — flag if off by more than 30%
hw_train = train_means.get('Highway', np.nan)
hw_pred  = pred_means .get('Highway', np.nan)
hw_ok = abs(hw_pred - hw_train) / hw_train < 0.30 if hw_train > 0 else False
check('Highway mean within 30% of train', hw_ok, f'(train={hw_train:.4f}, pred={hw_pred:.4f})')

Reference vs predicted mean by RoadType:
             train_mean  pred_mean_test
Highway        0.610756        0.433504
Residential    0.057209        0.059471
Street         0.273164        0.360564
PASS  Highway mean within 30% of train  (train=0.6108, pred=0.4335)


## 4. Predicted demand by qhour (should ramp up then plateau)

In [4]:
print('Mean prediction by qhour_of_day:')
print(joined.groupby('qhour_of_day')['demand'].mean())

Mean prediction by qhour_of_day:
qhour_of_day
9     0.112897
10    0.083975
11    0.080509
12    0.080772
13    0.075610
14    0.075033
15    0.078352
16    0.079184
17    0.075550
18    0.076162
19    0.076313
20    0.078175
21    0.074278
22    0.076455
23    0.075917
24    0.076642
25    0.078277
26    0.078263
27    0.077275
28    0.078573
29    0.077192
30    0.077808
31    0.079494
32    0.080317
33    0.080442
34    0.077708
35    0.078262
36    0.080760
37    0.081417
38    0.078595
39    0.080380
40    0.082851
41    0.085961
42    0.085064
43    0.085834
44    0.085832
45    0.086430
46    0.082985
47    0.084585
48    0.082931
49    0.082819
50    0.080008
51    0.079193
52    0.079980
53    0.078094
54    0.075908
55    0.074838
Name: demand, dtype: float64


## 5. Saturation correction sanity

In [5]:
fire_test  = np.load(os.path.join(PRED_DIR, 'sat_fire_mask_test.npy'))
sat_test_p = pd.read_parquet(os.path.join(PRED_DIR, 'saturation_prob_test.parquet'))
sat_test_p = sat_test_p.set_index('Index')['sat_prob_test']

blend = json.load(open(os.path.join(BASE, 'blend_weights.json')))
print('Saturation used:', blend['use_saturation'])
print(f'Fire mask test count: {int(fire_test.sum())}')

# Compute count of test predictions >= 0.99 (predicted-saturating)
high_preds = (sub['demand'] >= 0.99).sum()
print(f'Test predictions >= 0.99: {high_preds}')
check('At least some predicted saturation on Highway/4-5-lane cells',
      high_preds > 0 if blend['use_saturation'] else True)

Saturation used: False
Fire mask test count: 846
Test predictions >= 0.99: 2
PASS  At least some predicted saturation on Highway/4-5-lane cells  


## 6. Leakage spot-check — modal features train-only?

In [6]:
# Re-derive train-only modal RoadType and compare with cleaning_v2 output
def safe_mode(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else np.nan

cl_train = pd.read_csv(os.path.join(PROCESSED, 'train_clean_v2.csv'))
train_only_modal = raw_train.groupby('geohash')['RoadType'].agg(safe_mode)
v2_modal = cl_train.groupby('geohash')['RoadType_modal'].first()
shared = train_only_modal.index.intersection(v2_modal.index)
mismatches = (train_only_modal[shared] != v2_modal[shared]).sum()
check('RoadType_modal matches train-only mode (0 mismatches expected)', mismatches == 0,
      f'({mismatches} mismatches on shared geohashes)')

PASS  RoadType_modal matches train-only mode (0 mismatches expected)  (0 mismatches on shared geohashes)


## 7. Leakage spot-check — Temperature train-only imputation

In [7]:
# Recompute train-only (day,hour) mean and apply to test rows where Temperature was missing
test_imp_lookup = json.load(open(os.path.join(PROCESSED, 'imputation_lookups.json')))
# Use it to verify a sample
cl_test = pd.read_csv(os.path.join(PROCESSED, 'test_clean_v2.csv'))
test_orig_nan = raw_test['Temperature'].isna()
imp_count = test_orig_nan.sum()
print(f'Test rows with originally-missing Temperature: {imp_count}')
print(f'After imputation, NaN in test_clean_v2 Temperature: {cl_test["Temperature"].isna().sum()}')
check('No NaN in cl_test.Temperature after imputation', cl_test['Temperature'].isna().sum() == 0)

Test rows with originally-missing Temperature: 1349
After imputation, NaN in test_clean_v2 Temperature: 0
PASS  No NaN in cl_test.Temperature after imputation  


## 8. OOF blend re-derivation

In [8]:
# Reload OOF parquets, redo NNLS, compare weights vs blend_weights.json
from scipy.optimize import nnls
BASE_MODELS = ['lgbm_huber','lgbm_mae','lgbm_quantile','moe_blended','moe_expert','moe_global']
train_features = pd.read_parquet(os.path.join(PROCESSED, 'train_features.parquet'))
oof = train_features[['Index','demand']].copy()
for m in BASE_MODELS:
    p = pd.read_parquet(os.path.join(OOF_DIR, f'{m}.parquet')).rename(columns={f'{m}_oof_pred': m})
    oof = oof.merge(p, on='Index', how='left')
X = oof[BASE_MODELS].values
y = oof['demand'].values
w_redo, _ = nnls(X, y)
expected_w = blend['weights']
diff = max(abs(w_redo[i] - expected_w[m]) for i, m in enumerate(BASE_MODELS))
print('Re-derived NNLS weights vs saved blend_weights.json:')
for m, w_recomp in zip(BASE_MODELS, w_redo):
    print(f'  {m:18s}  saved={expected_w[m]:.5f}  redo={w_recomp:.5f}  diff={abs(w_recomp - expected_w[m]):.2e}')
check('NNLS weights reproducible', diff < 1e-6, f'(max diff {diff:.2e})')

Re-derived NNLS weights vs saved blend_weights.json:
  lgbm_huber          saved=0.23042  redo=0.23042  diff=0.00e+00
  lgbm_mae            saved=0.00000  redo=0.00000  diff=0.00e+00
  lgbm_quantile       saved=0.05722  redo=0.05722  diff=0.00e+00
  moe_blended         saved=0.00000  redo=0.00000  diff=0.00e+00
  moe_expert          saved=0.25305  redo=0.25305  diff=0.00e+00
  moe_global          saved=0.46986  redo=0.46986  diff=0.00e+00
PASS  NNLS weights reproducible  (max diff 0.00e+00)


## 9. Final summary

In [9]:
print('\n=== FINAL VERIFICATION SUMMARY ===')
print(f'submission.csv: {len(sub)} rows, {len(sub.columns)} cols, range [{sub["demand"].min():.5f}, {sub["demand"].max():.5f}]')
print(f'Estimated competition score (val_fold OOF R² × 100):')
val_r2 = blend['val_fold_r2']
best_metric = val_r2['blend_with_sat'] if blend['use_saturation'] else val_r2['blend_no_sat']
print(f'  blend (no sat)   : {val_r2["blend_no_sat"] * 100:.3f}')
print(f'  blend (with sat) : {val_r2["blend_with_sat"] * 100:.3f}')
print(f'  best (selected)  : {best_metric * 100:.3f}')
print('\nReady to upload to HackerEarth.')


=== FINAL VERIFICATION SUMMARY ===
submission.csv: 41778 rows, 2 cols, range [0.00354, 0.99965]
Estimated competition score (val_fold OOF R² × 100):
  blend (no sat)   : 96.720
  blend (with sat) : 96.688
  best (selected)  : 96.720

Ready to upload to HackerEarth.
